# Sailor2 Task 1 → Task 2 on Kaggle

Enable Internet and select **GPU T4 ×2**. Attach private `VGEC 2026_v1.json`, `VGEC_classifier.json`, and `VGEC_balanced.json` files from the same audited dataset build. Run All trains the existing Sailor2 Task 1 baseline, generates train-only out-of-fold hints, then trains alpha's Task 2 corrector. The models are independently initialized QLoRA adapters; Task 2 prompts contain the source sentence plus Task 1's predicted label and error-type description, not transferred weights or gold labels. Predicted class 0 never bypasses correction.

**Private GitHub access:** create a fine-grained token restricted to `rquq/GECC_VLSP_26` with **Contents: Read-only**. In Kaggle **Add-ons → Secrets**, add `GITHUB_TOKEN` and enable notebook access before Run All. Each bootstrap run clones the branch into a fresh source directory without putting the token in URLs or saved Git config. Old source edits and outputs are untouched. This does not download datasets: attach the private Kaggle Dataset separately.

`configs/baseline.json` retains pinned Sailor2-8B-Chat, NF4 QLoRA, rank 16 / alpha 32 / dropout 0.05, seed 42, five epochs, LR 5e-5 and effective batch eight. The unchanged audited data has 504 training rows (312 retained originals + 192 synthetic rows) and 81 original validation rows. A fresh fit selects its own best epoch by validation macro F1; it does not reuse the local trained adapter or guarantee identical GPU numerics.

Task 1 retains five epochs, LR 5e-5, effective batch eight, seed 42, rank 16. Its five OOF fits use three epochs each and exclude connected source/parent/template groups. Task 2 uses the configured Sailor2 recipe: eight epochs, LR 2e-5, effective batch eight, rank 16, response-only JSON supervision, and selection by error-only GLEU+, then exact match, then clean preservation.

Both stages and every OOF fit use two-process DDP: one NF4 replica on each T4, synchronized adapter gradients, per-GPU microbatch one and four accumulation steps for global effective batch eight. T4 uses FP16. Epoch validation and generation run on rank 0; the other rank waits. Two GPUs are required by default; this is data parallelism for training throughput, not a split model or a promise of 2× speed.

Inputs are discovered under `/kaggle/input`; set explicit paths below if filenames are ambiguous. Outputs use `/kaggle/output`, linked to `/kaggle/working/output` for notebook-version retention. Keep datasets, notebook versions, checkpoints and text-containing outputs private.

**Save Version:** use **Save & Run All** for an unattended run. Training dependencies are installed in a separate Python environment that reuses Kaggle's CUDA PyTorch. The notebook kernel and its preloaded NumPy are untouched; no kernel restart is required. Package-conflict warnings for unrelated preinstalled apps may still appear; the fresh training interpreter verifies the actual training imports before launch.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import signal
import sys

# Use PyTorch; avoid importing optional TensorFlow/Flax stacks.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")

REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "kaggle-linux"
PROJECT_SUBDIR = "."  # The project is at repository root.
CLASSIFICATION_CONFIG = "configs/baseline.json"
CLASSIFICATION_OVERRIDES = {}  # Optional baseline experiment overrides.
CORRECTION_CONFIG = "configs/correction.json"
CORRECTION_OVERRIDES = {}  # Optional Task 2 experiment overrides.
ORIGINAL_DATA = None  # e.g. "/kaggle/input/my-private-dataset/VGEC 2026_v1.json"
CLASSIFIER_DATA = None  # e.g. "/kaggle/input/my-private-dataset/VGEC_classifier.json"
CORRECTION_DATA = None  # e.g. "/kaggle/input/my-private-dataset/VGEC_balanced.json"
USE_HF_SECRET = False
SOURCE_ROOT = Path("/kaggle/working")
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
print("Branch:", BRANCH, "| configs:", CLASSIFICATION_CONFIG, CORRECTION_CONFIG)



In [ ]:
# Git runs as a subprocess; notebook execution needs no kernel restart.
# Always clone into a fresh directory; old source edits and outputs are untouched.
from urllib.parse import urlsplit
from contextlib import contextmanager
import tempfile
import os

@contextmanager
def github_git():
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    token = None
    with tempfile.TemporaryDirectory() as temporary:
        if urlsplit(REPO_URL).hostname == "github.com":
            from kaggle_secrets import UserSecretsClient
            try:
                token = UserSecretsClient().get_secret("GITHUB_TOKEN")
            except Exception:
                raise RuntimeError("Add GITHUB_TOKEN in Kaggle Secrets and enable notebook access. "
                                   "The private repository requires Contents: Read-only permission.") from None
            if not token:
                raise RuntimeError("GITHUB_TOKEN is empty; add a repository-read token in Kaggle Secrets.")
            askpass = Path(temporary) / "git-askpass.sh"
            askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) printf "%s\\n" x-access-token ;;\n'
                               '*) printf "%s\\n" "$GECC_GITHUB_TOKEN" ;;\nesac\n', encoding="utf-8")
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), GECC_GITHUB_TOKEN=token)

        def run(*args, cwd=None, check=True):
            result = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd,
                                    env=env, capture_output=True, text=True)
            if token:
                result.stderr = result.stderr.replace(token, "[REDACTED]")
                result.stdout = result.stdout.replace(token, "[REDACTED]")
            if check and result.returncode:
                detail = result.stderr.strip()
                raise RuntimeError("Git repository access failed. Check Kaggle Internet and GITHUB_TOKEN "
                                   "repository access (Contents: Read-only). " + detail)
            return result

        try:
            yield run
        finally:
            env.pop("GECC_GITHUB_TOKEN", None)
            token = None

if urlsplit(REPO_URL).username is not None:
    raise ValueError("Do not embed GitHub credentials in REPO_URL.")

with github_git() as git:
    remote = git("ls-remote", "--exit-code", "--heads", "--", REPO_URL,
                 f"refs/heads/{BRANCH}", check=False)
    if remote.returncode == 2:
        raise RuntimeError(f"Branch {BRANCH!r} does not exist on GitHub. Push it or select a published BRANCH.")
    if remote.returncode != 0:
        raise RuntimeError("Cannot read the Git repository. Enable Kaggle Internet and grant GITHUB_TOKEN "
                           "access to this private repository with Contents: Read-only permission. "
                           f"Git exit code {remote.returncode}: {remote.stderr.strip()}")
    source_dir = Path(tempfile.mkdtemp(prefix="gecc-two-phase-source-", dir=SOURCE_ROOT))
    git("clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
        "--", REPO_URL, str(source_dir))
PROJECT_DIR = (source_dir / PROJECT_SUBDIR).resolve()
if not PROJECT_DIR.is_relative_to(source_dir.resolve()):
    raise ValueError("PROJECT_SUBDIR must stay inside the cloned repository.")
for filename in ["requirements-kaggle.txt", "model/classification.py", "model/correction.py",
                 "model/two_phase.py", CLASSIFICATION_CONFIG, CORRECTION_CONFIG]:
    if not (PROJECT_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing {filename} in {PROJECT_DIR}; check the selected branch and PROJECT_SUBDIR.")
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH, "project_subdir": PROJECT_SUBDIR,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=source_dir, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))



In [ ]:
# Keep the notebook kernel untouched: Save Version cannot pause for a restart.
# Reuse Kaggle's CUDA torch through system-site-packages; install training pins locally.
from importlib.metadata import version
import tempfile

installed_torch = version("torch")
TRAINING_ENV = Path(tempfile.mkdtemp(prefix="gecc-training-env-"))
subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(TRAINING_ENV)], check=True)
TRAINING_PYTHON = str(TRAINING_ENV / "bin/python")
constraints = TRAINING_ENV / "constraints.txt"
constraints.write_text(f"torch=={installed_torch}\n", encoding="utf-8")
subprocess.run([TRAINING_PYTHON, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements-kaggle.txt"),
                "--constraint", str(constraints)], cwd=PROJECT_DIR, check=True)
subprocess.run([TRAINING_PYTHON, "-c",
                "import numpy, scipy, sklearn, torch, transformers, peft, accelerate, bitsandbytes; "
                "from transformers import AutoModelForCausalLM; from sklearn.metrics import roc_curve; "
                f"assert torch.__version__ == {installed_torch!r}; "
                "print('Training imports passed; torch', torch.__version__, 'CUDA', torch.version.cuda)"], check=True)
print("Isolated training environment ready; no kernel restart needed.")



In [ ]:
if USE_HF_SECRET:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

def apply_overrides(target, overrides):
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(target.get(key), dict):
            apply_overrides(target[key], value)
        else:
            target[key] = value

config = json.loads((PROJECT_DIR / CLASSIFICATION_CONFIG).read_text(encoding="utf-8"))
correction_config = json.loads((PROJECT_DIR / CORRECTION_CONFIG).read_text(encoding="utf-8"))
apply_overrides(config, CLASSIFICATION_OVERRIDES)
apply_overrides(correction_config, CORRECTION_OVERRIDES)
config["data"].update(path=ORIGINAL_DATA, augmentation_path=CLASSIFIER_DATA,
                      correction_path=CORRECTION_DATA, input_root="/kaggle/input")
correction_config["data"].update(path=CORRECTION_DATA, input_root="/kaggle/input")
for stage in (config, correction_config):
    stage["run"]["output_root"] = "/kaggle/output"
sys.path.insert(0, str(PROJECT_DIR))
from model.runtime import load_config, resolve_data

# /kaggle/working is retained by Save Version; expose the requested output path.
PERSISTED_OUTPUT = Path("/kaggle/working/output")
PERSISTED_OUTPUT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = Path("/kaggle/output")
if not OUTPUT_ROOT.exists() and not OUTPUT_ROOT.is_symlink():
    OUTPUT_ROOT.symlink_to(PERSISTED_OUTPUT, target_is_directory=True)
if OUTPUT_ROOT.resolve() != PERSISTED_OUTPUT.resolve():
    raise RuntimeError("/kaggle/output must link to /kaggle/working/output so Save Version retains artifacts.")
original_path = resolve_data(config)
mixed_config = json.loads(json.dumps(config))
mixed_config["data"].update(path=config["data"]["augmentation_path"],
                           filename=config["data"]["augmentation_filename"])
classifier_path = resolve_data(mixed_config)
correction_path = resolve_data(correction_config)
for data_path in (original_path, classifier_path, correction_path):
    if not data_path.is_relative_to(Path("/kaggle/input")):
        raise ValueError(f"Pipeline input must be an attached Kaggle dataset: {data_path}")
config["data"].update(path=str(original_path), augmentation_path=str(classifier_path),
                      correction_path=str(correction_path))
correction_config["data"]["path"] = str(correction_path)
CONFIG_PATH = PERSISTED_OUTPUT / f"classification-config-{RUN_ID}.json"
CORRECTION_CONFIG_PATH = PERSISTED_OUTPUT / f"correction-config-{RUN_ID}.json"
for path, stage in ((CONFIG_PATH, config), (CORRECTION_CONFIG_PATH, correction_config)):
    path.write_text(json.dumps(stage, ensure_ascii=False, indent=2), encoding="utf-8")
load_config(CONFIG_PATH, "classification")
load_config(CORRECTION_CONFIG_PATH, "correction")
OUTPUT_DIR = OUTPUT_ROOT / f"sailor2-two-phase-{RUN_ID}"
command = [TRAINING_PYTHON, "-m", "model.two_phase",
           "--classification-config", str(CONFIG_PATH), "--correction-config", str(CORRECTION_CONFIG_PATH),
           "--output-dir", str(OUTPUT_DIR)]


In [ ]:
# Train Task 1, generate OOF hints, then train Task 2; each saved adapter is reloaded.
LOG_PATH = OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}.log"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
with LOG_PATH.open("w", encoding="utf-8") as log_file:
    log_file.write(json.dumps(SOURCE_INFO, ensure_ascii=False) + "\n")
    log_file.flush()
    with subprocess.Popen(command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1,
                          start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log_file.write(line)
                log_file.flush()
            returncode = process.wait()
        except BaseException:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


In [ ]:
from IPython.display import JSON, FileLink, display
for filename in ["classification/metrics.json", "correction/metrics.json", "pipeline_manifest.json"]:
    print(filename)
    display(JSON(json.loads((OUTPUT_DIR / filename).read_text(encoding="utf-8"))))
import shutil
ARCHIVE_PATH = Path(shutil.make_archive(str(PERSISTED_OUTPUT / OUTPUT_DIR.name) + "-artifacts", "zip", OUTPUT_DIR))
display(FileLink(str(ARCHIVE_PATH)))
display(FileLink(str(LOG_PATH.resolve())))
print("Task 1 classifier:", OUTPUT_DIR / "classification/classifier")
print("Task 2 corrector:", OUTPUT_DIR / "correction/adapter")
print("Save a private notebook version to retain /kaggle/working/output.")


## Saved pipeline use

The run retains both selected adapters, their tokenizers, resolved configs, train/validation records, predictions, metrics, epoch histories and source manifest. `classification/hints.json` records the connected OOF folds. Validation selection is exploratory on the same holdout, not an independent test score. Artifacts and logs are retained under `/kaggle/working/output`.

Run inference on new records with both adapters; every row reaches Task 2:

```bash
python -m model.pipeline_predict --classifier /kaggle/output/RUN/classification/classifier \
  --adapter /kaggle/output/RUN/correction/adapter --data /kaggle/input/DATA/input.json \
  --output /kaggle/output/predictions.json
```

For a direct Kaggle terminal run, create a training environment, retain the output link as above, then run:

```bash
python -m venv --system-site-packages /tmp/gecc-training
/tmp/gecc-training/bin/python -m pip install -r requirements-kaggle.txt
/tmp/gecc-training/bin/python -m model.two_phase --classification-config configs/baseline.json \
  --correction-config configs/correction.json
```
